In [1]:
import sys
sys.path.insert(0, "src")
import numpy as np
import pandas as pd
from kai.models.ssc import GRBShock
from kai.inference.ssc_likelihood import (
    SSCLikelihood, JointLikelihood,
    mock_grb211211a_xrt_sed, ssc_priors_grb211211a
)
from kai.data.loaders import load_gw170817_photometry
from kai.inference.likelihood import KilonovaLikelihood

/home/jean/miniconda3/lib/python3.13/site-packages/bilby/core/likelihood.py:83: FutureWarning: <class 'kai.inference.likelihood.KilonovaLikelihood'> log_likelihood or log_likelihood_ratio method does not accept 'parameters' as an argument. This is deprecated behaviour  and will be removed in Bilby version 3. See https://bilby-dev.github.io/bilby/parameters for more details.
  warnings.warn(
/home/jean/miniconda3/lib/python3.13/site-packages/bilby/core/likelihood.py:83: FutureWarning: <class 'kai.inference.likelihood.KilonovaLikelihoodWithSystematics'> log_likelihood or log_likelihood_ratio method does not accept 'parameters' as an argument. This is deprecated behaviour  and will be removed in Bilby version 3. See https://bilby-dev.github.io/bilby/parameters for more details.
  warnings.warn(
/home/jean/miniconda3/lib/python3.13/site-packages/bilby/core/likelihood.py:83: FutureWarning: <class 'kai.inference.ssc_likelihood.SSCLikelihood'> log_likelihood or log_likelihood_ratio method doe

In [2]:
# ── SSC likelihood test ───────────────────────────────────────────────────────
shock = GRBShock(
    eiso=1e52, density=0.01, tstart=1e4, tstop=2e4,
    redshift=0.076, scenario='ISM'
)
ssc_data = mock_grb211211a_xrt_sed()
print("Mock XRT data:")
print(ssc_data)

ssc_lk = SSCLikelihood(data=ssc_data, shock=shock)
ssc_lk.summary()

Mock XRT data:
   energy_eV           sed       sed_err instrument
0      500.0  4.352753e-12  6.529129e-13        XRT
1     1000.0  5.000000e-12  7.500000e-13        XRT
2     2000.0  5.743492e-12  8.615238e-13        XRT
3     5000.0  6.898648e-12  1.034797e-12        XRT
4     8000.0  7.578583e-12  1.136787e-12        XRT
SSCLikelihood
  Free params      : ['log10_eta_e', 'log10_ebreak', 'alpha2', 'log10_ecut', 'log10_B']
  Data points      : 5
  Energy range     : 5.00e+02 -- 8.00e+03 eV
  Sigma floor      : 10%
  Absorption method: 2
GRBShock (ISM)
  Eiso         = 1.00e+52 erg
  density      = 1.000e-02 cm^-3
  t_obs        = 15000.0 s
  Gamma        = 15.55
  R            = 8.694e+17 cm
  E_shock/vol  = 7.266e-03 erg/cm^3
  redshift     = 0.076
  D_L          = 347.788 Mpc


/home/jean/miniconda3/lib/python3.13/site-packages/bilby/core/likelihood.py:127: FutureWarning: Setting non-trivial parameters for <class 'kai.inference.ssc_likelihood.SSCLikelihood'>. This is deprecated behaviour  and will be removed in Bilby version 3. See https://bilby-dev.github.io/bilby/parameters for more details.
  warnings.warn(msg, FutureWarning)


In [3]:
# Test at GRB 190829A Night 1 params
test_params = {
    'log10_eta_e':  -0.04,
    'log10_ebreak': -1.513,
    'alpha2':        3.15,
    'log10_ecut':    1.7,
    'log10_B':      -0.448,
}
for k, v in test_params.items():
    ssc_lk.parameters[k] = v

ln_l = ssc_lk.log_likelihood()
print(f"\nSSC log_likelihood = {ln_l:.4f}")
print(f"finite?            = {np.isfinite(ln_l)}")


/home/jean/miniconda3/lib/python3.13/site-packages/bilby/core/likelihood.py:113: FutureWarning: Parameter attribute queried for <class 'kai.inference.ssc_likelihood.SSCLikelihood'>. This is deprecated behaviour  and will be removed in Bilby version 3. See https://bilby-dev.github.io/bilby/parameters for more details.
  warnings.warn(msg, FutureWarning)



SSC log_likelihood = -163672073.5196
finite?            = True


In [4]:
# ── Joint likelihood test ─────────────────────────────────────────────────────
kn_data = load_gw170817_photometry(bands=["r", "J"])
kn_lk   = KilonovaLikelihood(data=kn_data, n_components=2, distance_mpc=40.0)
joint   = JointLikelihood(kn_lk, ssc_lk)
joint.summary()

print(f"\nJoint parameter space ({len(joint.parameters)} params):")
for k in joint.parameters:
    print(f"  {k}")

JointLikelihood
  Total parameters : 11
  KN parameters    : ['blue_mej', 'blue_vej', 'blue_kappa', 'red_mej', 'red_vej', 'red_kappa']
  SSC parameters   : ['log10_eta_e', 'log10_ebreak', 'alpha2', 'log10_ecut', 'log10_B']
KilonovaLikelihood
  Components : ['blue', 'red']
  Free params: ['blue_mej', 'blue_vej', 'blue_kappa', 'red_mej', 'red_vej', 'red_kappa']
  Data points: 16
  Bands      : ['J', 'r']
  Distance   : 40.0 Mpc
  Sigma floor: 0.1 mag
SSCLikelihood
  Free params      : ['log10_eta_e', 'log10_ebreak', 'alpha2', 'log10_ecut', 'log10_B']
  Data points      : 5
  Energy range     : 5.00e+02 -- 8.00e+03 eV
  Sigma floor      : 10%
  Absorption method: 2
GRBShock (ISM)
  Eiso         = 1.00e+52 erg
  density      = 1.000e-02 cm^-3
  t_obs        = 15000.0 s
  Gamma        = 15.55
  R            = 8.694e+17 cm
  E_shock/vol  = 7.266e-03 erg/cm^3
  redshift     = 0.076
  D_L          = 347.788 Mpc

Joint parameter space (11 params):
  blue_mej
  blue_vej
  blue_kappa
  red_mej
  

/home/jean/miniconda3/lib/python3.13/site-packages/bilby/core/likelihood.py:127: FutureWarning: Setting non-trivial parameters for <class 'kai.inference.likelihood.KilonovaLikelihood'>. This is deprecated behaviour  and will be removed in Bilby version 3. See https://bilby-dev.github.io/bilby/parameters for more details.
  warnings.warn(msg, FutureWarning)
/home/jean/miniconda3/lib/python3.13/site-packages/bilby/core/likelihood.py:113: FutureWarning: Parameter attribute queried for <class 'kai.inference.likelihood.KilonovaLikelihood'>. This is deprecated behaviour  and will be removed in Bilby version 3. See https://bilby-dev.github.io/bilby/parameters for more details.
  warnings.warn(msg, FutureWarning)
/home/jean/miniconda3/lib/python3.13/site-packages/bilby/core/likelihood.py:113: FutureWarning: Parameter attribute queried for <class 'kai.inference.ssc_likelihood.SSCLikelihood'>. This is deprecated behaviour  and will be removed in Bilby version 3. See https://bilby-dev.github.io/b